# Flower Detector Training (Oxford 102 Flowers)

Companion notebook to `train_flowers102.ipynb`. Fine-tunes a small, pretrained
**YOLO detector** (single class: "flower") so `api/predict.py` can locate
flower region(s) in a photo *before* classifying them -- this helps with
cluttered backgrounds and photos with multiple blooms of the same species,
which the classifier alone struggles with.

This is a separate, optional model. If you skip this notebook entirely,
the app keeps working exactly as before (whole-image classification) --
`api/predict.py` falls back automatically when `flower_detector.onnx` is
missing.

Runs for free on **Colab's T4 GPU**: Runtime -> Change runtime type -> T4 GPU.

## 1. Clone the repo and install extra dependencies

Set `BRANCH` to whichever branch has this notebook's companion scripts
(`prepare_detection_data.py`, `train_detector.py`, `export_detector_onnx.py`)
-- if you're running this before merging into `main`, that's your feature
branch name.

In [ ]:
REPO_URL = "https://github.com/hindh00/flower-recognition.git"  # <-- replace if needed
BRANCH = "object-detection-layer"  # <-- change to "main" once this is merged

!git clone -b $BRANCH $REPO_URL repo
%cd repo/training

!pip install -q ultralytics scikit-learn


## 2. Download data and derive detection labels

Downloads Oxford 102's segmentation archive (separate from the images/labels
`train_flowers102.ipynb` downloads) and derives one bounding box per image
from each trimap's foreground extent, writing an Ultralytics-ready dataset
under `DATA_ROOT/detection`.

Reuses the same Drive-cached `DATA_ROOT` pattern as the classifier notebook
so the ~330MB image download is shared/cached across both notebooks.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/flowers102_data"

from prepare_detection_data import build_yolo_dataset

dataset_yaml = build_yolo_dataset(data_root=DATA_ROOT, out_dir=f"{DATA_ROOT}/detection")
print(f"Dataset config: {dataset_yaml}")


## 3. Verify mask-to-box derivation

`mask_to_bbox()` classifies foreground pixels as `red_channel > blue_channel`
(Oxford 102's trimaps are blue-dominant background / red-white-dominant
foreground). Before trusting this across all 8,189 images, sanity-check it
visually on a handful of real samples -- especially multi-petal/hibiscus-like
flowers, which are the motivating case for this whole detector.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

label_dir = Path(f"{DATA_ROOT}/detection/labels/train")
image_dir = Path(f"{DATA_ROOT}/detection/images/train")
sample_labels = random.sample(list(label_dir.glob("*.txt")), 8)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, label_path in zip(axes.flat, sample_labels):
    img_path = image_dir / f"{label_path.stem}.jpg"
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    cls, xc, yc, bw, bh = map(float, label_path.read_text().split())
    x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
    x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
    draw = ImageDraw.Draw(img)
    draw.rectangle([x1, y1, x2, y2], outline="lime", width=4)
    ax.imshow(img)
    ax.set_title(label_path.stem)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("If any box looks wildly wrong (empty, covers the whole image, or misses the flower),")
print("adjust the r > b threshold in prepare_detection_data.py's mask_to_bbox() before training.")


## 4. Train the detector

Single class ("flower"), fine-tuned from a pretrained `yolo11n.pt` (COCO
weights, auto-downloaded on first use). A nano model keeps inference fast and
CPU-friendly for the deployed Vercel function.

In [ ]:
from train_detector import train_detector

detector_checkpoint = train_detector(
    data=str(dataset_yaml),
    model="yolo11n.pt",
    epochs=50,
    imgsz=640,
)
print(f"Best checkpoint: {detector_checkpoint}")


## 5. Sanity-check predictions

Run the trained detector on a few real validation images and look at the
boxes directly -- Ultralytics' own `results.png`/`val_batch*_pred.jpg` in the
run directory are also worth a look for aggregate metrics (mAP50/mAP50-95).

In [ ]:
import glob

from ultralytics import YOLO

detector = YOLO(str(detector_checkpoint))
sample_images = sorted(glob.glob(f"{DATA_ROOT}/detection/images/val/*.jpg"))[:6]
results = detector.predict(sample_images, imgsz=640, conf=0.25, iou=0.45)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, result in zip(axes.flat, results):
    ax.imshow(result.plot()[..., ::-1])  # BGR -> RGB
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. Export to ONNX (NMS baked in)

Bakes non-max suppression into the ONNX graph (`nms=True`) so
`api/predict.py` never has to hand-roll it -- just reads already-filtered
boxes. Runs a PyTorch-vs-ONNXRuntime parity check on real validation images
before trusting the export; if this assertion fails, do not ship the file.

In [ ]:
from export_detector_onnx import export

export(
    checkpoint=str(detector_checkpoint),
    out_path="../api/model/flower_detector.onnx",
    imgsz=640,
    sample_glob=f"{DATA_ROOT}/detection/images/val/*.jpg",
)


## 7. Get the results back into your repo

The exported file now lives at `api/model/flower_detector.onnx`. From this
Colab session:

```python
%cd /content/repo
!git add api/model/flower_detector.onnx
!git commit -m "Add trained flower detector"
!git push origin HEAD
```

(`git push origin HEAD` pushes to whichever branch you cloned, so this is
safe to run before merging.) You'll need to authenticate the push (a GitHub
personal access token works as the password prompt, or set up `git config`
with credentials first).

`api/predict.py` picks up `flower_detector.onnx` automatically the next time
it's deployed -- no code changes needed, since the detection pipeline was
already wired in ahead of time and only activates once this file exists.